In [ ]:
!pip install -q ultralytics

In [ ]:
from pathlib import Path
import os

# project name to search for
PROJECT_NAME = 'euro-detector-v2'

def find_project_dir(project_name=PROJECT_NAME):
    candidates = []

    # 1) if notebook/script is already launched from project root
    cwd = Path.cwd().resolve()
    if cwd.name == project_name:
        candidates.append(cwd)
    if (cwd / 'dataset' / 'data.yaml').exists():
        candidates.append(cwd)

    # 2) search upwards from cwd
    for p in [cwd, *cwd.parents]:
        if p.name == project_name and (p / 'dataset' / 'data.yaml').exists():
            candidates.append(p)

    # 3) common local places
    local_roots = [cwd, cwd.parent, Path.home(), Path('/content')]
    for root in local_roots:
        if root.exists():
            try:
                for p in root.rglob(project_name):
                    if p.is_dir() and (p / 'dataset' / 'data.yaml').exists():
                        candidates.append(p.resolve())
            except Exception:
                pass

    # 4) google drive if available
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        try:
            for p in drive_root.rglob(project_name):
                if p.is_dir() and (p / 'dataset' / 'data.yaml').exists():
                    candidates.append(p.resolve())
        except Exception:
            pass

    # unique while preserving order
    unique = []
    seen = set()
    for p in candidates:
        s = str(p)
        if s not in seen:
            seen.add(s)
            unique.append(p)

    assert unique, f'Project folder with dataset/data.yaml not found: {project_name}'
    return unique[0]

PROJECT_DIR = find_project_dir()
DATA_YAML = PROJECT_DIR / 'dataset' / 'data.yaml'
RUNS_DIR = PROJECT_DIR / 'runs'

print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_YAML =', DATA_YAML)
print('RUNS_DIR =', RUNS_DIR)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    project=str(RUNS_DIR),
    name='train'
)

In [ ]:
from ultralytics import YOLO

best_candidates = sorted(
    [p for p in RUNS_DIR.rglob('best.pt') if 'weights' in p.parts],
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

assert best_candidates, f'No best.pt found inside: {RUNS_DIR}'

best_path = best_candidates[0]
print('Using checkpoint:', best_path)

best_model = YOLO(str(best_path))
metrics = best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=640
)

print('mAP50-95:', metrics.box.map)
print('mAP50:', metrics.box.map50)
print('mAP75:', metrics.box.map75)
print('Precision:', metrics.box.mp)
print('Recall:', metrics.box.mr)